# COMM117 Week 6 Workshop — Post-training (SFT/DPO/RLHF) + More Pre-training (Scaling & PEFT)



**How to use:** Try the TODOs first. Each TODO is followed by a **Model answer** cell.


## Part 0 — Setup (do not edit)

In [ ]:
# Part 0 — Setup (do not edit)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cpu"
print("torch", torch.__version__, "device", device)


## Part 1 — Scaling & compute budgeting (Lecture 9)

These are short quantitative checks you should be able to do in interviews/exams.

### Exercise 1.1 — Effective batch size (accumulation)

You can only fit a **microbatch** of `B=8` sequences on a GPU, but you want an **effective batch** of `B_eff=64`.

**TODO**
1. Compute the number of accumulation steps.
2. If your loss is *averaged over the microbatch*, what scaling do you apply before `backward()`?

Write your answers as Python variables: `accum_steps` and `loss_scale`.


In [ ]:
# Model answer
accum_steps = 64 // 8
loss_scale = 1 / accum_steps   # multiply loss by this before backward() (equivalently, divide by accum_steps)
accum_steps, loss_scale


### Exercise 1.2 — Warmup + cosine decay LR schedule

Implement a function `lr_at(step, base_lr=3e-4, warmup=50, max_steps=500)`:

- linear warmup for the first `warmup` steps
- cosine decay to 0 after warmup until `max_steps`

Return a float learning rate.


In [ ]:
# Model answer
def lr_at(step, base_lr=3e-4, warmup=50, max_steps=500):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    # cosine over remaining region
    denom = (max_steps - warmup)
    if denom <= 0:
        return base_lr
    tau = (step - warmup) / denom
    tau = min(max(tau, 0.0), 1.0)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * tau))

(lr_at(0), lr_at(49), lr_at(50), lr_at(499))


## Part 2 — Supervised Fine-Tuning (SFT) mechanics

We’ll build a tiny toy “chat” dataset and implement **loss masking** (train only on assistant tokens).

In [ ]:
# A toy tokenizer/vocab for demonstration (character-level-ish)
vocab = ["<pad>", "<bos>", "<eos>", "<usr>", "<asst>"] + list("abcdefghijklmnopqrstuvwxyz ?!.")
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for s,i in stoi.items()}

def encode(text):
    return [stoi[c] for c in text]

def decode(ids):
    return "".join(itos[i] for i in ids)

len(vocab)


### Exercise 2.1 — Build labels with masking

We represent a chat sample as:

`<bos> <usr> user_text <asst> assistant_text <eos>`

**TODO**
Write `build_sft_example(user_text, assistant_text)` returning:
- `input_ids` (list[int])
- `labels` (list[int]) where tokens belonging to **user** part are masked to `-100`, and tokens belonging to assistant response (including `<asst>` token is usually masked; here **mask everything before the first assistant character**) are supervised.

Rule for this exercise:
- Mask `<bos> <usr> user_text <asst>` to `-100`
- Supervise assistant characters and `<eos>` (i.e., labels equal to token id)

Test it on: user=`"hi?"`, assistant=`"ok."`


In [ ]:
# Model answer
def build_sft_example(user_text, assistant_text):
    seq = ["<bos>", "<usr>"] + list(user_text) + ["<asst>"] + list(assistant_text) + ["<eos>"]
    input_ids = [stoi[t] if t in stoi else stoi[t] for t in seq]  # tokens are in vocab
    # mask everything up to and including <asst>
    labels = [-100] * len(input_ids)
    # find first assistant character position (the token right after <asst>)
    asst_idx = seq.index("<asst>")
    for i in range(asst_idx + 1, len(seq)):
        labels[i] = input_ids[i]
    return input_ids, labels

inp, lab = build_sft_example("hi?", "ok.")
decode(inp), lab, len(inp)


### Exercise 2.2 — Masked cross-entropy loss

Given model logits `logits` of shape `[T, V]` and labels `labels` of shape `[T]` with `-100` masked positions,
implement `masked_ce_loss(logits, labels)`.

Use PyTorch `F.cross_entropy(..., ignore_index=-100)`.


In [ ]:
# Model answer
def masked_ce_loss(logits, labels):
    return F.cross_entropy(logits, labels, ignore_index=-100)

T, V = 6, len(vocab)
logits = torch.randn(T, V)
labels = torch.tensor([-100, -100, 3, 4, -100, 5])
masked_ce_loss(logits, labels).item()


## Part 3 — Preference optimisation: DPO loss (Lecture 10)

We’ll implement the **Direct Preference Optimisation (DPO)** objective on toy log-probabilities.

DPO uses pairs `(prompt, chosen, rejected)` and compares the model’s relative preference vs a reference model.

### Exercise 3.1 — Implement DPO loss (scalar version)

Let:
- `pi_logp_c`, `pi_logp_r` be the *policy* model log-probs for chosen/rejected completions
- `ref_logp_c`, `ref_logp_r` be the *reference* model log-probs
- `beta` be the temperature (e.g., 0.1 to 0.5)

DPO loss for one pair can be written:

\[
\mathcal{L}= -\log \sigma\left( \beta \left[(\pi_c-\pi_r) - (ref_c-ref_r)\right] \right)
\]

**TODO:** implement `dpo_loss(pi_logp_c, pi_logp_r, ref_logp_c, ref_logp_r, beta=0.1)` returning a scalar tensor.


In [ ]:
# Model answer
def dpo_loss(pi_logp_c, pi_logp_r, ref_logp_c, ref_logp_r, beta=0.1):
    delta_pi = pi_logp_c - pi_logp_r
    delta_ref = ref_logp_c - ref_logp_r
    z = beta * (delta_pi - delta_ref)
    return -F.logsigmoid(z)

pi_c = torch.tensor(-1.0)
pi_r = torch.tensor(-2.0)
ref_c = torch.tensor(-1.2)
ref_r = torch.tensor(-1.6)
dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=0.5).item()


### Exercise 3.2 — Interpret the sign

Assume `beta>0`. When does DPO loss get **smaller**?

**TODO:** Fill in the sentence:

> The DPO loss decreases when the policy prefers the **chosen** completion over the rejected completion **more than** the reference does, meaning `(pi_c - pi_r) ____ (ref_c - ref_r)`.


In [ ]:
# Model answer
relation = ">"
relation


## Part 4 — Parameter-efficient fine-tuning (PEFT): LoRA

We implement a minimal LoRA linear layer and verify the parameter count reduction.

### Exercise 4.1 — LoRA parameter counting

For a linear layer `d -> d` with `d=4096` and LoRA rank `r=8`:
- Full fine-tune trains `d*d` parameters (ignoring bias).
- LoRA trains `2*d*r` parameters.

Compute both and the percentage.


In [ ]:
# Model answer
d = 4096
r = 8
full = d * d
lora = 2 * d * r
pct = 100 * lora / full
full, lora, pct


### Exercise 4.2 — Implement a LoRA linear layer

Implement `LoRALinear(in_features, out_features, r, alpha)` such that:
- base weight `W` is frozen (no grad)
- trainable low-rank matrices `A` (r x in), `B` (out x r)
- forward: `x @ W^T + (alpha/r) * x @ A^T @ B^T`

**TODO:** finish the class so that only LoRA parameters require gradients.


In [ ]:
# Model answer
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, alpha=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha

        # frozen base weight
        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

        # LoRA params (trainable)
        self.A = nn.Parameter(torch.zeros(r, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, r))
        # common init: A random, B zeros so initial delta is ~0
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

        self.scaling = alpha / r

    def forward(self, x):
        base = F.linear(x, self.weight)                 # x @ W^T
        delta = (x @ self.A.t()) @ self.B.t()           # x @ A^T @ B^T
        return base + self.scaling * delta

layer = LoRALinear(16, 32, r=4, alpha=4)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
trainable, frozen


## Part 5 — Quick concept checks (open answers )

Use these as mini revision prompts.

### Exercise 5.1 — Base model vs assistant model
**Question:** What does post-training change relative to a base pretrained model?

**Answer:** Post-training (e.g., SFT + preference optimisation like RLHF/DPO + safety tuning) changes **behavioral policy**: instruction-following, helpfulness, refusal style, and conversational alignment—without changing the fundamental pretraining objective that built linguistic competence.

---

### Exercise 5.2 — When is warmup especially important?
**Question:** Name two situations where warmup helps stability.

**Answer:** (i) Training deep Transformers with Adam/AdamW at non-trivial LR; (ii) Post-LN or other fragile architectures; warmup prevents early large updates before moments/activations settle.

---

### Exercise 5.3 — RLHF vs DPO
**Question:** Give one advantage of DPO over PPO-style RLHF.

**Answer:** DPO is simpler and often more stable/cheaper: it learns directly from preference pairs without training a reward model and without on-policy RL rollouts.
